# Demo 6: Q transfer with gaze-select-restrict: the goal-head *query* carries "what is left to do"

**Question.** Demo 5 showed that at the pivot the goal heads re-read the sub-goal from the task span (the *keys and
values*). What does the pivot's *query* contribute? The hypothesis: the query encodes the **progress**, an abstract
"these sub-tasks are done, find the next one", and the task span supplies the content. If so, a query taken from
another prompt should select whichever sub-task is *still pending in the destination*, not copy the source's
content or its list position.

**Setup (order-diff).** SRC and DST share two sub-tasks a, b; SRC has a third c, DST a third d:

```
SRC: (1) a  (2) c  (3) b     prefill {a, b}   -> SRC writes c   (c is listed 2nd)
DST: (1) a  (2) b  (3) d     prefill {a}      -> DST writes b   (native)
```

We transplant SRC's goal-head query at the pivot (`c =`) into DST. SRC's query says "a and b are done". Resolved
against DST's list, the only task with a and b done is **d**. A literal content copy predicts c (not even in DST's
spec); a positional copy ("the 2nd-listed") predicts b.

**Intervention (gaze-select-restrict, GSR).** (1) *Select*: run DST with SRC's query in the 250 goal heads at the pivot
and score each listed DST task by the heads' value-weighted attention from the pivot onto its description; pick the
top one. (2) *Restrict*: in a separate run **without** the query transfer, from the pivot on the goal heads may only
attend to the selected task's span, the pivot and the generated tokens. Control: the same with 250 random heads.

**Paper.** §5.1.2 (`fig:q-transfer`), App. `app:transfer:q` (`fig:q-transfer-schema`, `fig:q-barcharts`,
`tab:q-matched`, `tab:q-diff`). This demo runs the paper's schema example.

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file.

**From the talk** (figures from `Goals_lab_talk.pptx`):

![Query transfer: how is the next operation decided? (§5.1.2)](figures/06_q_transfer_setup.png)

*Query transfer: how is the next operation decided? (§5.1.2)*

![Query transfer step by step: select, then generate (App. I.2).](figures/06_q_transfer_steps.png)

*Query transfer step by step: select, then generate (App. I.2).*

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
huggingface_hub.utils.disable_progress_bars()

repo root: /disk/u/arnab/Codes/Projects/goals


## Config
Defaults are those of `scripts/nrgs/q_transfer_eval.py`, the paper's configuration: `--patch-heads G250
--readout-heads G250 --alphas 2 --n-samples 32 --seed 0`, pivot = the frontier's `{var} =`, letters naming
(`a`, `b`, `c` in the order the lines are written), clean generations of 96 tokens, the GSR generation 64 tokens.
α = 2 is an OV boost of the patched heads that applies **only to the patched GoalScope readout**, not to the selection
or the generation.

**Why this instance.** It is the default of `scripts/nrgs/q_transfer_example_attention.py`, the example drawn in the
paper's schema (`fig:q-transfer-schema`): `int`, Python→Python, order-diff, a = `is_prime`, b = `is_positive`,
c = `count_set_bits`, d = `last_digit`. In the reported run it is a causal HIT (writes d) with a shifted readout under
the goal heads and a miss under the control.

In [2]:
CUDA     = '7'
MODEL    = 'Qwen/Qwen2.5-Coder-32B-Instruct'
G250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json'
C250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/control/control250__laterhalf.json'
DATATYPE, LANG, VARIANT = 'int', 'py-py', 'order_diff'
KEY_CONTAINS = 'dst=is_prime-is_positive-last_digit'   # q_transfer_example_attention.py default
ALPHA    = 2.0
SEED     = 0

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model
`src.nrgs.functional.setup` loads the model (bf16, eager). The eval script's module (`scripts/nrgs/q_transfer_eval.py`)
is imported only for its constants and helpers; its `main()` (which writes to `results/`) is not called.

In [3]:
import json, hashlib, html as _html
import numpy as np, torch
from rich.console import Console
from rich.table import Table
from rich.text import Text
from rich.markup import escape
from IPython.display import HTML, display

import src.nrgs.data as DATA
import src.nrgs.functional as FUNC
import src.nrgs.eval as E
import scripts.nrgs.q_transfer_eval as QTE
from src.utils import env_utils
from src.utils.experiment_utils import set_seed

console = Console(width=140, highlight=False)
torch.set_grad_enabled(False)

ctx = FUNC.setup(MODEL, device_map={'': 0})
mt = ctx.mt
goal_heads, ctrl_heads = QTE._resolve_heads(G250), QTE._resolve_heads(C250)
print(f'goal heads: {len(goal_heads)} | control heads: {len(ctrl_heads)} (overlap {len(set(goal_heads) & set(ctrl_heads))})')

2026-10-30 21:34:45 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
goal heads: 250 | control heads: 250 (overlap 0)


## 1. The contrastive pair
`DATA.diff_trigger_setups` draws the eval's 32 instances for this cell with the same per-cell seed as
`q_transfer_eval.run_config`; we pick the schema instance by key.

In [4]:
sl, dl = QTE.LANG_TRANSFERS[LANG]
tag = f"{DATATYPE.replace('[', '(').replace(']', ')')}__{sl}-{dl}"
cfg_seed = (SEED + int(hashlib.md5(f'{VARIANT}/{tag}'.encode()).hexdigest()[:8], 16)) % (2**32)
set_seed(cfg_seed)
setups = DATA.diff_trigger_setups(DATATYPE, variant=VARIANT, n_samples=32, seed=cfg_seed, src_lang=sl, dst_lang=dl)
inst = next(i for i in setups if KEY_CONTAINS in QTE._inst_key(i))
KEY = QTE._inst_key(inst)
A, B = inst.src_progress
C, D = inst.src_target, inst.diff_target
LET = {A: 'a', B: 'b', C: 'c', D: 'd'}
COL = {A: 'white', B: 'yellow', C: 'cyan', D: 'magenta'}
print(KEY)

t = Table(title='task lists and progress', show_lines=True)
t.add_column(''); t.add_column('SRC'); t.add_column('DST')
fmt = lambda ps: '\n'.join(f'({k}) [{COL[p.id]}]{LET[p.id]}: {escape(p.description)}[/]' for k, p in enumerate(ps, 1))
t.add_row('listed', fmt(inst.src_order), fmt(inst.dst_order))
t.add_row('prefilled (done)', ', '.join(LET[x] for x in inst.src_progress), ', '.join(LET[x] for x in inst.dst_progress))
t.add_row('pending', ', '.join(LET[p.id] for p in inst.src_order if p.id not in inst.src_progress),
          ', '.join(LET[p.id] for p in inst.dst_order if p.id not in inst.dst_progress))
t.add_row('writes next (clean)', f'[cyan]c[/] = {escape(inst.src_target_desc)}', f'[yellow]b[/] = {escape(inst.dst_next_desc)}')
console.print(t)
console.print(f'[bold]predictions for the patched DST[/]:  progress (DIFF) -> [magenta]d[/] = {escape(inst.expected_next_desc)}'
              f'  |  literal copy -> [cyan]c[/]  |  positional copy / no effect -> [yellow]b[/]')

order_diff|src=is_prime-count_set_bits-is_positive|dst=is_prime-is_positive-last_digit|prog_src=is_prime-is_positive|prog_dst=is_prime


                                                  task lists and progress                                                   
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                     ┃ SRC                                                        ┃ DST                                   ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ listed              │ (1) a: whether the number is prime                         │ (1) a: whether the number is prime    │
│                     │ (2) c: the number of set bits in the binary representation │ (2) b: whether the number is positive │
│                     │ (3) b: whether the number is positive                      │ (3) d: the last decimal digit         │
├─────────────────────┼────────────────────────────────────────────────────────────┼───────────────────────────────────────┤
│ prefilled (done)    │ a, b                                                       │ a                                     │
├─────────────────────┼────────────────────────────────────────────────────────────┼───────────────────────────────────────┤
│ pending             │ c                                                          │ b, d                                  │
├─────────────────────┼────────────────────────────────────────────────────────────┼───────────────────────────────────────┤
│ writes next (clean) │ c = the number of set bits in the binary representation    │ b = whether the number is positive    │
└─────────────────────┴────────────────────────────────────────────────────────────┴───────────────────────────────────────┘

predictions for the patched DST:  progress (DIFF) -> d = the last decimal digit  |  literal copy -> c  |  positional copy / no effect -> b

The two prompts as the model sees them (user turn onward). Both end at the pivot `{var} =`, the `=` of the
frontier line; that is where SRC's query is read and where it is written into DST.

In [5]:
mat = DATA.materialize(inst)
def show_prompt(prompt, title):
    user = prompt[prompt.index('<|im_start|>user'):]
    console.rule(title); console.print(Text(user, style='dim'), Text('  <- pivot', style='bold red'))
show_prompt(mat['src_prompt'], 'SRC prompt (query donor)')
show_prompt(mat['dst_prompt'], 'DST prompt (the run we patch)')

───────────────────────────────────────────────────────── SRC prompt (query donor) ─────────────────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given an integer `n`, computes:
(1) whether the number is prime
(2) the number of set bits in the binary representation
(3) whether the number is positive
<|im_end|>
<|im_start|>assistant
<code>
def func(n: int):
    a = n > 1 and all(n % i != 0 for i in range(2, int(n**0.5) + 1))
    b = (n > 0)
    c =   <- pivot

────────────────────────────────────────────────────── DST prompt (the run we patch) ───────────────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given an integer `n`, computes:
(1) whether the number is prime
(2) whether the number is positive
(3) the last decimal digit
<|im_end|>
<|im_start|>assistant
<code>
def func(n: int):
    a = n > 1 and all(n % i != 0 for i in range(2, int(n**0.5) + 1))
    b =   <- pivot

## 2. Run gaze-select-restrict through the goal heads
`FUNC.run_q_transfer_intervention` is exactly what the eval script calls per instance. It returns the clean SRC / DST
generations, the selection scores and the selected task, the GSR generation, three GoalScope readouts (SRC pivot, DST
pivot, DST pivot under the query patch + α OV boost), and the goal heads' value-weighted attention from the DST pivot
onto the task list, clean and under the query patch (the latter is the selection pass).

In [6]:
raw = FUNC.run_q_transfer_intervention(mt, inst, goal_heads, readout_heads=goal_heads, max_new=QTE.MAX_NEW,
                                      alpha=ALPHA, compute_readout=True, gsr_max_new=QTE.GSR_MAX_NEW)
first = lambda s: s.strip().splitlines()[0] if s.strip() else ''
for k in ('gen_src', 'gen_dst'):
    console.print(f'clean {k[4:].upper()} continuation of the pivot: [green]{escape(first(raw[k]))}[/]')
console.print('selection scores:', {LET[k]: round(v, 3) for k, v in raw['cl_select_scores'].items()},
              '-> selected', f"[bold]{LET[raw['cl_selected']]}[/] ({raw['cl_selected']})")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


pos=150 ' =' => I'm checking if a number is positive and counting the number of 1s in its binary representation.

pos=136 ' =' => I'm checking if the number is greater than zero.

pos=136 ' =' => I'm checking the last digit of the number by using the modulus operator (`% 10`).

clean SRC continuation of the pivot: bin(n).count('1')

clean DST continuation of the pivot: n > 0

selection scores:
{'a': 0.051, 'b': 0.074, 'd': 0.306}
-> selected d (last_digit)

## 3. Where the pivot looks: the selection pass
The attention of the 250 goal heads (value-weighted, mean over heads) from the DST pivot `=` onto each token of the
three task descriptions, as in the paper's schema (`notebooks/fig_n_table_generators/q_transfer_schema.py`): green
shading ∝ attention on one shared scale, and the per-task *gaze* (mean over the description's tokens) on the right.
The left panel is the clean DST pivot, the right one the same pivot with SRC's query (selection pass).

In [7]:
def gaze_panels(raw, inst, heads_name):
    dst = raw['dst_prompt']
    pos = raw['task_positions']
    att = {'clean': dict(zip(pos, raw['attn_clean'])), 'patched': dict(zip(pos, raw['attn_patched']))}
    items = {p.id: list(range(*FUNC.token_span(dst, p.description))) for p in inst.task.predicates}
    score = {m: {pid: float(np.mean([att[m][q] for q in ps])) for pid, ps in items.items()} for m in att}
    amax = max(att[m][q] for m in att for ps in items.values() for q in ps)
    gmax = max(max(s.values()) for s in score.values())
    ids = FUNC.prepare_input(prompts=dst, tokenizer=mt)['input_ids'][0].tolist()
    tok = lambda q: _html.escape(mt.tokenizer.decode([ids[q]]))
    def panel(m, title):
        best = max(score[m], key=score[m].get)
        rows = ''
        for k, p in enumerate(inst.task.predicates, 1):
            spans = ''.join(f'<span style="background:rgba(63,143,90,{0.04 + 0.66 * min(1, att[m][q] / amax):.3f});'
                            f'border-radius:2px">{tok(q)}</span>' for q in items[p.id])
            g = score[m][p.id]
            bar = (f'<span style="display:inline-block;width:{110 * g / gmax:.0f}px;height:11px;background:#3f8f5a;'
                   f'opacity:{0.35 + 0.65 * g / gmax:.2f};border-radius:2px"></span> '
                   f'<span style="font-weight:{600 if p.id == best else 400}">{g:.3f}</span>')
            rows += (f'<tr><td style="color:#d23c35;font-family:serif;font-size:16px;padding-right:6px">{LET[p.id]}</td>'
                     f'<td style="font-family:monospace;white-space:pre">({k}){spans}</td>'
                     f'<td style="padding-left:14px;text-align:left">{bar}</td></tr>')
        return (f'<div style="flex:1;background:#f7f6f3;padding:10px 12px;border-radius:6px;color:#1f1e1c">'
                f'<div style="font-weight:600;margin-bottom:6px">{title}</div><table>{rows}</table></div>')
    display(HTML(f'<div style="display:flex;gap:14px">'
                 f'{panel("clean", f"(ii) clean DST pivot · {heads_name}")}'
                 f'{panel("patched", f"(iii) selection pass: SRC query at the DST pivot · {heads_name}")}</div>'))
    return score

score_goal = gaze_panels(raw, inst, 'goal 250')

a,(1) whether the number is prime,0.065
b,(2) whether the number is positive,0.239
d,(3) the last decimal digit,0.062
a,(1) whether the number is prime,0.051
b,(2) whether the number is positive,0.074
d,(3) the last decimal digit,0.306


In the clean DST pivot the goal heads look mostly at b, the task DST is about to write (0.239 vs ≈0.06 for a and d).
With SRC's query ("a and b are done") the gaze moves to **d** (0.306), the one task still pending under that progress,
and away from b (0.074). It does not move to the 2nd-listed task (b) nor to anything like c: the query is resolved
against DST's own list.

### The restrict step
From the pivot on, the goal heads may only read the positions in `cl_restrict_keep`: the selected task's description,
the pivot itself (and every token generated after it). Everything else in the prompt is masked for those 250 heads
only; all other heads, and the residual stream, are untouched. Below, the DST user turn with the tokens the goal heads
can still see highlighted.

In [8]:
ids = FUNC.prepare_input(prompts=raw['dst_prompt'], tokenizer=mt)['input_ids'][0].tolist()
keep = set(raw['cl_restrict_keep'])
start = max(i for i, t in enumerate(ids) if t == mt.tokenizer.convert_tokens_to_ids('<|im_start|>') and i < min(keep))
txt = Text()
for i in range(start, len(ids)):
    txt.append(mt.tokenizer.decode([ids[i]]), style='bold green on #e7f3eb' if i in keep else 'dim')
console.print(txt)

<|im_start|>user
Write a Python program that implements a function `func` which, given an integer `n`, computes:
(1) whether the number is prime
(2) whether the number is positive
(3) the last decimal digit
<|im_end|>
<|im_start|>assistant
<code>
def func(n: int):
    a = n > 1 and all(n % i != 0 for i in range(2, int(n**0.5) + 1))
    b =

## 4. Control: the same GSR through 250 random heads
As in the reported control (`--patch-heads C250 --readout-heads C250 --no-readout`): the query of the random heads is
transplanted, *their* gaze selects, and *they* are restricted. No readout.

In [9]:
raw_ctrl = FUNC.run_q_transfer_intervention(mt, inst, ctrl_heads, readout_heads=ctrl_heads, max_new=QTE.MAX_NEW,
                                           alpha=ALPHA, compute_readout=False, gsr_max_new=QTE.GSR_MAX_NEW)
console.print('control selection scores:', {LET[k]: round(v, 3) for k, v in raw_ctrl['cl_select_scores'].items()},
              '-> selected', f"[bold]{LET[raw_ctrl['cl_selected']]}[/]")
score_ctrl = gaze_panels(raw_ctrl, inst, 'control 250')

control selection scores:
{'a': 0.018, 'b': 0.021, 'd': 0.054}
-> selected d

a,(1) whether the number is prime,0.041
b,(2) whether the number is positive,0.055
d,(3) the last decimal digit,0.026
a,(1) whether the number is prime,0.018
b,(2) whether the number is positive,0.021
d,(3) the last decimal digit,0.054


The random heads' gaze also tips toward d under the transplanted query, but it is weak and nearly flat
(0.018 / 0.021 / 0.054, vs 0.306 for the goal heads), so the selection is not informative here. The decisive test is
the restrict step, judged below.

## 5. Judge
The eval script's oracles (Claude, cached on disk). **causal** = does the GSR continuation's first computation
implement d? (`judge_causal`; BOTH counts as a miss.) **joint** = one exclusive verdict among b (native), c (literal
SRC content), d (progress), neither (`judge_joint`). **literal c** = `classify_concept` against c.
**readout** = does the patched GoalScope readout express d rather than b (`eval_readouts`)?

In [10]:
def judge(raw, readout=True):
    causal = E.judge_causal(inst, raw['gen_patched'], model_name=QTE.ORACLE_MODEL)
    joint = E.judge_joint(inst, raw['gen_patched'], model_name=QTE.ORACLE_MODEL)
    lit = E.classify_concept(raw['gen_patched'], inst.src_target_desc, inst.dst_next_desc, inst.task_desc,
                             model_name=QTE.ORACLE_MODEL,
                             kind='the code the model generated after the pivot; judge ONLY the first computation')
    ro = (E.eval_readouts(inst, raw['ro_src'], raw['ro_dst'], raw['ro_patched'], model_name=QTE.ORACLE_MODEL)
          if readout else None)
    return dict(causal=causal, joint=joint, literal_c=lit['label'] == 'src', readout=ro)

verd = {'goal 250': judge(raw), 'control 250': judge(raw_ctrl, readout=False)}
JL = {'native': '[yellow]b (native)[/]', 'src_target': '[cyan]c (literal)[/]', 'diff_target': '[magenta]d (progress)[/]',
      'neither': 'neither'}
RL = {'src': '[magenta]d[/]', 'dst': '[yellow]b[/]'}

t = Table(title=f'd = "{inst.expected_next_desc}"   b = "{inst.dst_next_desc}"   c = "{inst.src_target_desc}"',
          show_lines=True)
t.add_column('run'); t.add_column('selected'); t.add_column('continuation of the pivot', max_width=40)
t.add_column('causal (d?)'); t.add_column('joint'); t.add_column('GoalScope readout at the pivot', max_width=46)
t.add_column('readout says')
ro0 = verd['goal 250']['readout']
t.add_row('clean SRC', '-', escape(first(raw['gen_src'])), '-', '-', escape(raw['ro_src'].strip()), RL.get(ro0['source'], ro0['source']))
t.add_row('clean DST', '-', escape(first(raw['gen_dst'])), '-', '-', escape(raw['ro_dst'].strip()), RL.get(ro0['dest'], ro0['dest']))
for name, r in (('goal 250', raw), ('control 250', raw_ctrl)):
    v = verd[name]
    hit = '[green]HIT[/]' if v['causal']['triggered'] else '[red]miss[/]'
    ro = v['readout']
    t.add_row(f'GSR: {name}', LET[r['cl_selected']], escape(first(r['gen_patched'])), hit, JL.get(v['joint']['label'], v['joint']['label']),
              escape((r['ro_patched'] or 'off').strip()),
              (RL.get(ro['patched'], ro['patched']) + ('  [green]shifted[/]' if ro['shifted'] else '')) if ro else 'off')
console.print(t)

      d = "the last decimal digit"   b = "whether the number is positive"   c = "the number of set bits in the binary representation"       
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ run              ┃ selected ┃ continuation of the pivot ┃ causal (d?) ┃ joint        ┃ GoalScope readout at the pivot     ┃ readout says ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ clean SRC        │ -        │ bin(n).count('1')         │ -           │ -            │ I'm checking if a number is        │ b            │
│                  │          │                           │             │              │ positive and counting the number   │              │
│                  │          │                           │             │              │ of 1s in its binary                │              │
│                  │          │                           │             │              │ representation.                    │              │
├──────────────────┼──────────┼───────────────────────────┼─────────────┼──────────────┼────────────────────────────────────┼──────────────┤
│ clean DST        │ -        │ n > 0                     │ -           │ -            │ I'm checking if the number is      │ b            │
│                  │          │                           │             │              │ greater than zero.                 │              │
├──────────────────┼──────────┼───────────────────────────┼─────────────┼──────────────┼────────────────────────────────────┼──────────────┤
│ GSR: goal 250    │ d        │ n % 10                    │ HIT         │ d (progress) │ I'm checking the last digit of the │ d  shifted   │
│                  │          │                           │             │              │ number by using the modulus        │              │
│                  │          │                           │             │              │ operator (`% 10`).                 │              │
├──────────────────┼──────────┼───────────────────────────┼─────────────┼──────────────┼────────────────────────────────────┼──────────────┤
│ GSR: control 250 │ d        │ n > 0                     │ miss        │ b (native)   │ off                                │ off          │
└──────────────────┴──────────┴───────────────────────────┴─────────────┴──────────────┴────────────────────────────────────┴──────────────┘

Through the goal heads, GSR turns DST's native `n > 0` (b) into `n % 10`: a causal HIT, the joint judge calls it
d (progress), not a literal copy of c and not the native b. The patched GoalScope readout shifts the same way, from
"checking if the number is greater than zero" to "checking the last digit". Restricting 250 random heads to the same
selected span leaves the continuation at `n > 0` (miss): the effect lives in the goal heads. (The clean SRC readout
verbalizes both b and c, the done and the next task; its label is only a sanity check, not part of the metric.)

## 6. Cross-check against the reported run
The cached records of this instance (`results/diff/<model>/diff_trigger/<run_tag>/order_diff/int__python-python/results.json`)
and the schema data (`results/diff/<model>/gsr_example/`, written by `q_transfer_example_attention.py`), then the
reported rates for this cell and pooled over all 32 cells (2 variants × 4 datatypes × 4 language pairs).

In [11]:
R = os.path.join(env_utils.DEFAULT_RESULTS_DIR, 'diff', MODEL.split('/')[-1], 'diff_trigger')
TAGS = {'goal 250': 'ph-goal_250__a2__piv-eq__cl-ppgazerestrict-anceq__seq',
        'control 250': 'ph-control250_250__a2__piv-eq__cl-ppgazerestrict-anceq__seq'}
ours = {'goal 250': raw, 'control 250': raw_ctrl}
t = Table(title='this notebook vs. the reported run (same instance)')
for c in ('run', 'paper causal', 'ours', 'same selection', 'same GSR gen?', f'{VARIANT}/{tag} causal',
          'pooled order-matched', 'pooled order-diff'):
    t.add_column(c)
for name, rt in TAGS.items():
    rec = next(r for r in json.load(open(os.path.join(R, rt, VARIANT, tag, 'results.json'))) if r['key'] == KEY)
    cell = json.load(open(os.path.join(R, rt, VARIANT, tag, 'summary.json')))
    pooled = {}
    for var in ('order_matched', 'order_diff'):
        ss = [json.load(open(os.path.join(R, rt, var, d, 'summary.json'))) for d in sorted(os.listdir(os.path.join(R, rt, var)))]
        n = sum(s['n'] for s in ss)
        pooled[var] = f"{sum(s['causal_trigger_rate'] * s['n'] for s in ss) / n:.3f} (n={n})"
    t.add_row(name, 'HIT' if rec['causal']['triggered'] else 'miss', 'HIT' if verd[name]['causal']['triggered'] else 'miss',
              str(rec['cl_selected'] == ours[name]['cl_selected']), str(rec['gen_patched'] == ours[name]['gen_patched']),
              f"{cell['causal_trigger_rate']:.3f}", pooled['order_matched'], pooled['order_diff'])
console.print(t)

import glob
ex = next(json.load(open(f)) for f in glob.glob(os.path.join(env_utils.DEFAULT_RESULTS_DIR, 'diff', MODEL.split('/')[-1],
                                                              'gsr_example', f'{VARIANT}__{tag}__*.json'))
          if json.load(open(f))['key'] == KEY)
console.print('schema data, selection scores:', {LET[k]: round(v, 3) for k, v in ex['scores_patched'].items()},
              '| ours:', {LET[k]: round(v, 3) for k, v in score_goal['patched'].items()})

                                             this notebook vs. the reported run (same instance)                                             
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃             ┃              ┃      ┃                ┃               ┃ order_diff/int__python-… ┃                      ┃                   ┃
┃ run         ┃ paper causal ┃ ours ┃ same selection ┃ same GSR gen? ┃ causal                   ┃ pooled order-matched ┃ pooled order-diff ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ goal 250    │ HIT          │ HIT  │ True           │ True          │ 0.594                    │ 0.871 (n=512)        │ 0.572 (n=512)     │
│ control 250 │ miss         │ miss │ True           │ True          │ 0.094                    │ 0.043 (n=512)        │ 0.029 (n=512)     │
└─────────────┴──────────────┴──────┴────────────────┴───────────────┴──────────────────────────┴──────────────────────┴───────────────────┘

schema data, selection scores:
{'a': 0.051, 'b': 0.074, 'd': 0.306}
| ours:
{'a': 0.051, 'b': 0.074, 'd': 0.306}

Our selection scores and GSR generations match the reported records of this instance exactly, for both the goal
and the control heads, and so do the HIT / miss verdicts. This instance is one of the hits: the cell's causal rate is
0.594, and pooled over the 32 cells the goal heads reach 0.871 (order-matched) / 0.572 (order-diff) against
0.043 / 0.029 for the control, the paper's numbers.

**The full result** (from the talk):

![Paper result: swapping the query moves the frontier about half the time (§5.1.2, Fig. 6).](figures/06_q_transfer_result.png)

*Paper result: swapping the query moves the frontier about half the time (§5.1.2, Fig. 6).*

## Where this goes in the paper / how to reproduce at scale
- **Commands**: `job_list/diff_eval/diff_trigger_gsr_eq_crosslang.json` + `diff_trigger_gaze_select_restrict_anc_eq.json`
  (together the 32 cells), e.g.
  ```
  python scripts/nrgs/q_transfer_eval.py --datatypes int --variants order_diff --lang-transfers py-py --tag-suffix __seq
  ```
  (`--tag-suffix __seq` is required to hit the reported dirs). Control: `diff_trigger_gsr_eq_control250.json` +
  `diff_trigger_gsr_eq_control250_rest.json` (`--patch-heads C250 --readout-heads C250 --no-readout`). Smoke:
  `--max-instances 2`. Records land in `results/diff/Qwen2.5-Coder-32B-Instruct/diff_trigger/<run_tag>/<variant>/<cell>/`;
  point `RESULTS_DIR` in `env.yml` to a scratch dir for a re-run.
- **Paper numbers**: pooled causal order-matched 0.871 (87%), order-diff 0.572 (57%); readout 0.668 / 0.412; control
  0.043 / 0.029 (≈4%).
- **Schema figure**: `python scripts/nrgs/q_transfer_example_attention.py --cuda 0` (this instance) then
  `python notebooks/fig_n_table_generators/q_transfer_schema.py`. Bar charts / tables:
  `notebooks/fig_n_table_generators/q_transfer.ipynb`.
- **Related**: KV transfer of the task span (Demo 5) and task span vs code-progress span (Demo 7).